<a href="https://colab.research.google.com/github/victoriashushpannikova/nlp_homeworks/blob/main/%D0%BA%D0%BE%D0%BF%D0%B8%D1%8F_fine_tuning_hw_(lab6_7).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

# Ячейка 1. Установка библиотек

In [ ]:
!pip install -q transformers datasets evaluate accelerate

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    pipeline
)
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


# Ячейка 2. Проверка GPU

In [ ]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

GPU available: True
GPU name: Tesla T4


# Ячейка 3. Загрузка датасета

In [ ]:
dataset = load_dataset("fancyzhx/ag_news")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

# Ячейка 4. Выбор модели и токенизатора

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Ячейка 5. Токенизация

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

# Ячейка 6. Data collator

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Ячейка 7. Метрика accuracy

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

# Ячейка 8. Параметры обучения

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_ag_news",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
)

# Ячейка 9. Trainer


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Ячейка 10. Обучение

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.195845,0.175991,0.943421
2,0.134900,0.190322,0.947368


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Ячейка 11. Accuracy на тесте

In [ ]:
test_results = trainer.evaluate()
print(test_results)
print("Test accuracy:", test_results["eval_accuracy"])

{'eval_loss': 0.21759656071662903, 'eval_accuracy': 0.9467105263157894, 'eval_runtime': 23.5657, 'eval_samples_per_second': 322.502, 'eval_steps_per_second': 20.156, 'epoch': 3.0}
Test accuracy: 0.9467105263157894


# Ячейка 12. Сохранение модели

In [ ]:
save_path = "./ag_news_model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ag_news_model/tokenizer_config.json', './ag_news_model/tokenizer.json')

# Ячейка 13. Pipeline для проверки

In [ ]:
classifier = pipeline("text-classification", model=save_path, tokenizer=save_path, truncation=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

# Ячейка 14. Тест на 3 новых новостях

In [16]:
news_samples = [
    "Apple announced a new AI-powered laptop with improved battery life and faster performance.",
    "The football team won the championship after a dramatic final match in overtime.",
    "Scientists discovered a new exoplanet that may have conditions suitable for life."
]

label_map = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

for text in news_samples:
    result = classifier(text)[0]
    pred_id = int(result["label"].split("_")[-1]) if "_" in result["label"] else int(result["label"])
    print("Text:", text)
    print("Predicted class:", label_map[pred_id])
    print("Confidence:", round(result["score"], 4))
    print()

Text: Apple announced a new AI-powered laptop with improved battery life and faster performance.
Predicted class: Sci/Tech
Confidence: 0.996

Text: The football team won the championship after a dramatic final match in overtime.
Predicted class: Sports
Confidence: 0.9944

Text: Scientists discovered a new exoplanet that may have conditions suitable for life.
Predicted class: Sci/Tech
Confidence: 0.9871

